## Healthcare Nursing Documentation Burden — Fabric IQ

### Generate Delta Tables & KQL Tables from Healthcare CSV Data

This notebook loads the 18 healthcare CSV files into:
- **Lakehouse** (Delta tables) — 6 dimension tables + 6 fact tables
- **Eventhouse** (KQL tables) — 6 event/streaming tables for Real-Time Intelligence

> **Prerequisites:**
> 1. Upload all CSV files from `datasets/healthcare_nursing_documentation/data/` to `/lakehouse/default/Files/healthcare_data/`
> 2. Attach a **Lakehouse** to this notebook
> 3. Create an **Eventhouse** and note the cluster URI and database name

In [ ]:
# Install Fabric IQ Ontology Accelerator Package
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

In [ ]:
import sempy.fabric as fabric
from notebookutils import mssparkutils

# ── Configuration ───────────────────────────────────────────────────────
csv_base_path = "/lakehouse/default/Files/healthcare_data"
lakehouse_schema = "dbo"  # replace if using lakehouse without schemas

# ── Dimension tables (Lakehouse Delta) ──────────────────────────────────
dimension_tables = [
    "dim_nurses",
    "dim_patients",
    "dim_hospital_units",
    "dim_documentation_types",
    "dim_medications",
    "dim_diagnoses",
]

# ── Fact tables bound to Lakehouse (Delta) ──────────────────────────────
lakehouse_fact_tables = [
    "fact_shifts",
    "fact_patient_encounters",
    "fact_care_plans",
    "fact_burnout_surveys",
    "fact_patient_satisfaction",
    "fact_documentation_quality",
]

print("Loading dimension tables into Lakehouse...")
for table_name in dimension_tables:
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"{csv_base_path}/{table_name}.csv")
    df.write.mode("overwrite").saveAsTable(f"{lakehouse_schema}.{table_name}")
    print(f"  ✓ {table_name} — {df.count()} rows")

print("\nLoading fact tables into Lakehouse...")
for table_name in lakehouse_fact_tables:
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"{csv_base_path}/{table_name}.csv")
    df.write.mode("overwrite").saveAsTable(f"{lakehouse_schema}.{table_name}")
    print(f"  ✓ {table_name} — {df.count()} rows")

print("\n✅ All 12 Lakehouse Delta tables created successfully.")

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties, DataFormat
import pandas as pd

# ── Eventhouse Configuration ────────────────────────────────────────────
eventhouse_cluster_uri = ""   # e.g. "https://<name>.<region>.kusto.fabric.microsoft.com"
eventhouse_database = ""      # your Eventhouse database name
access_token = mssparkutils.credentials.getToken(eventhouse_cluster_uri)

# ── Event tables → Eventhouse (KQL) ─────────────────────────────────────
# Maps CSV file name → KQL table name
eventhouse_tables = {
    "fact_documentation_events": "documentation_events",
    "fact_medication_administration": "medication_administration",
    "fact_vital_signs": "vital_signs",
    "fact_assessments": "assessments",
    "fact_ehr_interactions": "ehr_interactions",
    "fact_handoff_reports": "handoff_reports",
}

# ── Real-Time Streaming tables → Eventhouse (KQL) ───────────────────────
streaming_tables = {
    "stream_rtls_location": "rtls_location",
    "stream_ehr_clickstream": "ehr_clickstream",
    "stream_nurse_call_events": "nurse_call_events",
    "stream_bcma_scans": "bcma_scans",
    "stream_clinical_alerts": "clinical_alerts",
}

# Merge both dictionaries for ingestion
eventhouse_tables.update(streaming_tables)

kcsb = KustoConnectionStringBuilder.with_aad_application_token_authentication(
    eventhouse_cluster_uri, access_token
)
kusto_client = KustoClient(kcsb)

print("Loading event and streaming data into Eventhouse KQL tables...")
for csv_name, kql_table in eventhouse_tables.items():
    csv_path = f"{csv_base_path}/{csv_name}.csv"
    pdf = spark.read.option("header", True).option("inferSchema", True).csv(csv_path).toPandas()

    # Create table schema from DataFrame columns
    type_map = {"int64": "long", "float64": "real", "object": "string", "datetime64[ns]": "datetime"}
    columns = ", ".join(
        f"['{col}']: {type_map.get(str(pdf[col].dtype), 'string')}" for col in pdf.columns
    )
    create_cmd = f".create-merge table {kql_table} ({columns})"
    kusto_client.execute_mgmt(eventhouse_database, create_cmd)

    # Ingest CSV data
    ingest_kcsb = KustoConnectionStringBuilder.with_aad_application_token_authentication(
        eventhouse_cluster_uri, access_token
    )
    ingest_client = QueuedIngestClient(ingest_kcsb)
    ingestion_props = IngestionProperties(
        database=eventhouse_database,
        table=kql_table,
        data_format=DataFormat.CSV,
    )
    ingest_client.ingest_from_dataframe(pdf, ingestion_properties=ingestion_props)
    print(f"  ✓ {kql_table} — {len(pdf)} rows ingested")

print("\n✅ All 11 Eventhouse KQL tables created and ingested successfully.")

### Data Summary

| Storage | Table | Source CSV | Record Count |
|---------|-------|-----------|--------------|
| Lakehouse | dim_nurses | dim_nurses.csv | 20 |
| Lakehouse | dim_patients | dim_patients.csv | 30 |
| Lakehouse | dim_hospital_units | dim_hospital_units.csv | 3 |
| Lakehouse | dim_documentation_types | dim_documentation_types.csv | 22 |
| Lakehouse | dim_medications | dim_medications.csv | 20 |
| Lakehouse | dim_diagnoses | dim_diagnoses.csv | 15 |
| Lakehouse | fact_shifts | fact_shifts.csv | 45 |
| Lakehouse | fact_patient_encounters | fact_patient_encounters.csv | 30 |
| Lakehouse | fact_care_plans | fact_care_plans.csv | 14 |
| Lakehouse | fact_burnout_surveys | fact_burnout_surveys.csv | 20 |
| Lakehouse | fact_patient_satisfaction | fact_patient_satisfaction.csv | 15 |
| Lakehouse | fact_documentation_quality | fact_documentation_quality.csv | 20 |
| Eventhouse | documentation_events | fact_documentation_events.csv | 130 |
| Eventhouse | medication_administration | fact_medication_administration.csv | 34 |
| Eventhouse | vital_signs | fact_vital_signs.csv | 38 |
| Eventhouse | assessments | fact_assessments.csv | 16 |
| Eventhouse | ehr_interactions | fact_ehr_interactions.csv | 49 |
| Eventhouse | handoff_reports | fact_handoff_reports.csv | 12 |
| **Eventhouse (Stream)** | **rtls_location** | **stream_rtls_location.csv** | **95** |
| **Eventhouse (Stream)** | **ehr_clickstream** | **stream_ehr_clickstream.csv** | **135** |
| **Eventhouse (Stream)** | **nurse_call_events** | **stream_nurse_call_events.csv** | **35** |
| **Eventhouse (Stream)** | **bcma_scans** | **stream_bcma_scans.csv** | **57** |
| **Eventhouse (Stream)** | **clinical_alerts** | **stream_clinical_alerts.csv** | **38** |

**Totals:** 12 Lakehouse Delta tables (254 rows) + 11 Eventhouse KQL tables (642 rows)